# 🚀 AIE4ML Tutorial 2: Branching + Permute + Multi-output

This short tutorial focuses on one model that stresses key architectural features:
- internal fanout / broadcasting after a shared dense layer
- `Permute((2, 1))` in one branch
- two output tensors
- ND dense path (Dense on rank-3 tensors)

## ⚙️ Setup & Imports

```bash
conda create -n aie4ml_env python=3.10 -y && conda activate aie4ml_env
pip install "tensorflow==2.15.*" "tensorflow-model-optimization==0.7.5" qkeras hls4ml
pip install aie4ml pyparsing ipykernel pydot graphviz
```
```

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Permute
from qkeras import QDense, QActivation, quantized_bits, quantized_relu
import hls4ml

from aie4ml.model import from_hls4ml
from aie4ml.report import report

## 1️⃣ The model

A shared trunk fans out to two branches. **Branch 1** applies a `Permute`; **Branch 2** is a
straight dense chain. Each branch has its own output. The input is quantized to **16 bits**;
weights and activations are int8. 

In [ ]:
TOKENS, FEATURES = 16, 64

def dense(name, units):
    return QDense(units, name=name,
                  kernel_quantizer=quantized_bits(8, 0, alpha=1),
                  bias_quantizer=quantized_bits(8, 2, alpha=1))

def build_model():
    inp = Input(shape=(TOKENS, FEATURES), name="in")
    x = QActivation(quantized_bits(16, 6), name="in_q16")(inp)      # 16-bit input

    x = dense("trunk_fc", FEATURES)(x)                              # shared trunk -> fanout
    x = QActivation(quantized_relu(8), name="trunk_relu")(x)

    b1 = dense("b1_fc1", FEATURES)(x)
    b1 = QActivation(quantized_relu(8), name="b1_relu")(b1)
    b1 = Permute((2, 1), name="b1_permute")(b1)
    b1 = dense("b1_fc2", FEATURES)(b1)
    b1 = QActivation(quantized_bits(8, 2), name="b1_out")(b1)

    b2 = dense("b2_fc1", FEATURES)(x)
    b2 = QActivation(quantized_relu(8), name="b2_relu1")(b2)
    b2 = dense("b2_fc2", FEATURES)(b2)
    b2 = QActivation(quantized_relu(8), name="b2_relu2")(b2)
    b2 = dense("b2_fc3", FEATURES)(b2)
    b2 = QActivation(quantized_bits(8, 2), name="b2_out")(b2)

    return Model(inp, [b1, b2], name="branching_model")

model = build_model()
model.compile(loss="mse")
model.summary()

## 2️⃣ Tuning directives: parallelism and placement

Directives are attached per layer in the hls4ml config.

**Parallelism** — `cas_num` independent cascade chains, each `cas_length` deep.
**Placement** — `{'row': r, 'col': c}` pins a kernel to a tile. 
Branch 2 is a straight chain with compatible layouts/partinioning, so its kernels can connect **directly** (avoiding the memory tiles). In this case specifically, each can read the memory bank its left neighbour wrote so we can pin them to **adjacent columns**, otherwise we have to keep an empty tile in between. The compiler's automatic placer can usually handle the placement for us.

In [ ]:
cfg = hls4ml.utils.config_from_keras_model(model, granularity="name")

for layer in ("trunk_fc", "b1_fc1", "b1_fc2", "b2_fc1", "b2_fc2", "b2_fc3"):
    cfg["LayerName"][layer]["parallelism"] = {"cas_num": 2, "cas_length": 1, "contract": "outer"}

for column, layer in zip((7, 8, 9), ("b2_fc1", "b2_fc2", "b2_fc3")):
    cfg["LayerName"][layer]["placement"] = {"row": 1, "col": column}

## 3️⃣ Convert and compile

Use the hls4ml frontend to convert to the AIE backend, then compile for the x86 simulator.

In [ ]:
aie_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=cfg,
    output_dir="proj_aie_t2", project_name="proj_aie_t2",
    backend="aie", batch_size=1, iterations=10,
)
aie_model.compile()

## 4️⃣ Bit-exact comparison with QKeras


In [ ]:
x = np.random.random((1, TOKENS, FEATURES)).astype(np.float32)

y_qkeras = model.predict(x, verbose=0)
y_aie = aie_model.predict(x, simulator="x86")

for name, out, ref in zip(("branch1", "branch2"), y_aie.values(), y_qkeras):
    print(f"{name}: max abs diff = {float(np.max(np.abs(np.asarray(out) - ref)))}")

## 5️⃣ Build for the AI Engine and profile

The x86 simulator checks the output numbers but not the timing. To get **cycles and latency**, we
compile the graph for the AI Engine and run the cycle-accurate simulator with profiling.

In [ ]:
aie_model.build()                                   # aiecompiler: compile for the AI Engine
y_aie_sim = aie_model.predict(x, simulator="aie")   # cycle-accurate simulation + profiling

for name, out, ref in zip(("branch1", "branch2"), y_aie_sim.values(), y_qkeras):
    print(f"{name}: max abs diff = {float(np.max(np.abs(np.asarray(out) - ref)))}")

## 6️⃣ The project report

`report` summarises the compiled project — how many kernels, how many AIE tiles and memory-tile
buffers, and (after running inference in '`aie`' mode) cycles and latency. 

In [ ]:
# Takes a model or a project directory -- report('proj_aie_t2') works the same.
report(aie_model)

## 💡 Takeaways

- One model exercised **fanout, a transpose, two outputs, ND dense, and a 16-bit input** 
- **Parallelism** (`cas_num` / `cas_length` / `contract`) trades AIE tiles for throughput.
- A **transpose** (`Permute`) is realised through a memory tile automatically;
- Tuning don't change the numerics — the result stays **bit-exact** with QKeras.

The same directives apply through the ONNX frontend, which is the more expressive and more fully
supported path in aie4ml.